# Assignment 1: Toán cho Học máy (MA1391)
## Nhiệm vụ C - Chương 5: Gradient và Ma trận Hesse
**Tập dữ liệu sử dụng:** Seeds Dataset (UCI Machine Learning Repository)
**Link dữ liệu:** https://archive.ics.uci.edu/dataset/236/seeds

In [ ]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

# Tải tập dữ liệu Seeds từ UCI Machine Learning Repository
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00236/seeds_dataset.txt'
try:
    df = pd.read_csv(url, sep='\s+', header=None)
except Exception as e:
    df = pd.read_csv('seeds_dataset.txt', sep='\s+', header=None)

# Chọn 2 đặc trưng đầu tiên làm x_n1, x_n2 và cột thứ 7 làm biến mục tiêu y_n
X_raw = df.iloc[:, :2].values
y = df.iloc[:, 7].values

N = X_raw.shape[0]
print(f'Kích thước tập dữ liệu N = {N} mẫu, số đặc trưng = 2')

### C1 & C2: Đạo hàm giải tích, Gradient và Ma trận Hesse
Xét hàm mất mát bình phương trên hai đặc trưng đã chọn:
$$L(w_1, w_2) = \frac{1}{N} \sum_{n=1}^{N} (w_1 x_{n1} + w_2 x_{n2} - y_n)^2$$

- **C1:** Đạo hàm riêng theo $w_1$ và $w_2$, biểu diễn gradient dưới dạng vectơ hàng $1 \times 2$.
- **C2:** Ma trận Hesse $\nabla^2 L$ và chứng minh không phụ thuộc $w$, kết luận tính lồi.

In [ ]:
# C3: Vẽ đồ thị tính toán của L cho trường hợp N = 2 và tính đạo hàm bằng chế độ ngược (Reverse-mode autodiff)
def computational_graph_demo():
    x1_val = X_raw[:2, 0]
    x2_val = X_raw[:2, 1]
    y_val = y[:2]
    
    w1 = torch.tensor(1.0, requires_grad=True)
    w2 = torch.tensor(1.0, requires_grad=True)
    
    # Các bước trong đồ thị tính toán cho N = 2
    pred_0 = w1 * x1_val[0] + w2 * x2_val[0]
    err_0 = pred_0 - y_val[0]
    sq_0 = err_0 ** 2
    
    pred_1 = w1 * x1_val[1] + w2 * x2_val[1]
    err_1 = pred_1 - y_val[1]
    sq_1 = err_1 ** 2
    
    L = (sq_0 + sq_1) / 2.0
    
    L.backward()
    
    print('--- C3: Kết quả đồ thị tính toán với N = 2 ---')
    print(f'Giá trị Loss L: {L.item():.4f}')
    print(f'Đạo hàm dL/dw1 (chế độ ngược): {w1.grad.item():.4f}')
    print(f'Đạo hàm dL/dw2 (chế độ ngược): {w2.grad.item():.4f}')

computational_graph_demo()

In [ ]:
# C4: Kiểm chứng C1 bằng vi phân tự động (torch.autograd) tại 3 điểm w khác nhau
def analytical_gradient(w1, w2, X, y):
    N = len(y)
    pred = w1 * X[:, 0] + w2 * X[:, 1]
    diff = pred - y
    grad_w1 = (2.0 / N) * np.sum(diff * X[:, 0])
    grad_w2 = (2.0 / N) * np.sum(diff * X[:, 1])
    return np.array([[grad_w1, grad_w2]])

test_points = [
    (0.0, 0.0),
    (1.5, -2.0),
    (0.5, 1.0)
]

print('--- C4: Kiểm chứng gradient giải tích và PyTorch autograd tại 3 điểm w ---')
for idx, (w1_val, w2_val) in enumerate(test_points, 1):
    grad_ana = analytical_gradient(w1_val, w2_val, X_raw, y)
    
    w1_t = torch.tensor(w1_val, dtype=torch.float64, requires_grad=True)
    w2_t = torch.tensor(w2_val, dtype=torch.float64, requires_grad=True)
    X_t = torch.tensor(X_raw, dtype=torch.float64)
    y_t = torch.tensor(y, dtype=torch.float64)
    
    pred_t = w1_t * X_t[:, 0] + w2_t * X_t[:, 1]
    L_t = torch.mean((pred_t - y_t) ** 2)
    L_t.backward()
    
    grad_torch = np.array([[w1_t.grad.item(), w2_t.grad.item()]])
    abs_error = np.abs(grad_ana - grad_torch)
    
    print(f'Điểm {idx} w = ({w1_val}, {w2_val}):')
    print(f'  - Giải tích: {grad_ana}')
    print(f'  - PyTorch  : {grad_torch}')
    print(f'  - Sai số tuyệt đối: {abs_error}\n')

In [ ]:
# C5: Khai triển Taylor bậc hai của L quanh điểm w = (0, 0)
def taylor_expansion_demo():
    w1_eval, w2_eval = 0.5, 0.5
    
    w1_0, w2_0 = 0.0, 0.0
    pred_0 = w1_0 * X_raw[:, 0] + w2_0 * X_raw[:, 1]
    L_0 = np.mean((pred_0 - y) ** 2)
    
    grad_0 = analytical_gradient(0.0, 0.0, X_raw, y)
    
    N = len(y)
    H_11 = (2.0 / N) * np.sum(X_raw[:, 0] ** 2)
    H_12 = (2.0 / N) * np.sum(X_raw[:, 0] * X_raw[:, 1])
    H_21 = H_12
    H_22 = (2.0 / N) * np.sum(X_raw[:, 1] ** 2)
    H = np.array([[H_11, H_12], [H_21, H_22]])
    
    w_vec = np.array([w1_eval, w2_eval])
    
    term_linear = np.dot(grad_0.flatten(), w_vec)
    term_quad = 0.5 * np.dot(w_vec, np.dot(H, w_vec))
    L_taylor = L_0 + term_linear + term_quad
    
    pred_actual = w1_eval * X_raw[:, 0] + w2_eval * X_raw[:, 1]
    L_actual = np.mean((pred_actual - y) ** 2)
    
    print('--- C5: Khai triển Taylor bậc hai quanh (0,0) ---')
    print(f'L(0,0) = {L_0:.4f}')
    print(f'Gradient tại (0,0) = {grad_0}')
    print(f'Ma trận Hesse H = \n{H}')
    print(f'Giá trị xấp xỉ Taylor tại w = ({w1_eval}, {w2_eval}): {L_taylor:.6f}')
    print(f'Giá trị thực tế L tại w = ({w1_eval}, {w2_eval}): {L_actual:.6f}')
    print(f'Sai số giữa Taylor và Thực tế: {np.abs(L_taylor - L_actual):.2e} (Chính xác tuyệt đối do L là đa thức bậc 2)')

taylor_expansion_demo()